In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch.nn as nn

from torchvision import models
from torchvision import datasets, transforms
from torchvision.transforms.functional import to_pil_image

from torch.utils.data import DataLoader

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
os.makedirs(
    "../outputs/vit/attention_rollout",
    exist_ok=True
)

os.makedirs(
    "../outputs/vit/attention_rollout/correct",
    exist_ok=True
)

os.makedirs(
    "../outputs/vit/attention_rollout/misclassified",
    exist_ok=True
)

In [ ]:
class CustomMultiheadAttention(
    nn.MultiheadAttention
):

    def __init__(self, *args, **kwargs):

        super().__init__(*args, **kwargs)

        self.attention_weights = None

    def forward(
        self,
        query,
        key,
        value,
        **kwargs
    ):

        attn_output, attn_weights = super().forward(

            query,
            key,
            value,

            need_weights=True,

            average_attn_weights=False
        )

        self.attention_weights = (
            attn_weights.detach()
        )

        return attn_output, attn_weights

In [ ]:
from torchvision.models import (
    vit_b_16,
    ViT_B_16_Weights
)

NUM_CLASSES = 3

model = vit_b_16(
    weights=ViT_B_16_Weights.DEFAULT
)

for block in model.encoder.layers:

    old_attn = block.self_attention

    new_attn = CustomMultiheadAttention(

        embed_dim=old_attn.embed_dim,

        num_heads=old_attn.num_heads,

        dropout=old_attn.dropout,

        batch_first=True
    )

    new_attn.load_state_dict(
        old_attn.state_dict()
    )

    block.self_attention = new_attn

model.heads.head = nn.Linear(

    model.heads.head.in_features,

    NUM_CLASSES
)

model.load_state_dict(

    torch.load(
        "../models/vit_b16.pth",
        map_location=device
    )
)

model.to(device)

model.eval()

print("Model loaded successfully!")

In [ ]:
class_names = [
    "COVID",
    "Normal",
    "Viral Pneumonia"
]

In [ ]:
transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5159, 0.5159, 0.5159],
        std=[0.2280, 0.2280, 0.2280]
    )
])

In [ ]:
def generate_attention_rollout(

    model,
    input_tensor
):

    with torch.no_grad():

        _ = model(input_tensor)

    all_attentions = []

    for block in model.encoder.layers:

        attn = (
            block.self_attention
            .attention_weights
        )

        # Shape:
        # [B, Heads, Tokens, Tokens]

        attn_mean = attn.mean(dim=1)

        all_attentions.append(
            attn_mean
        )

    rollout = torch.eye(

        all_attentions[0].size(-1)

    ).to(device)

    for attn in all_attentions:

        # Residual connection
        attn = attn + torch.eye(

            attn.size(-1)

        ).to(device)

        # Normalize rows
        attn = attn / attn.sum(

            dim=-1,
            keepdim=True
        )

        # Recursive multiplication
        rollout = torch.matmul(
            attn,
            rollout
        )

    # CLS token attention
    mask = rollout[
        0,
        0,
        1:
    ]

    # Patch grid
    grid_size = int(
        mask.size(0) ** 0.5
    )

    mask = mask.reshape(
        grid_size,
        grid_size
    )

    mask = mask.detach().cpu().numpy()

    # Resize
    mask = cv2.resize(
        mask,
        (224, 224)
    )

    # NORMALIZATION
    mask = mask - mask.min()

    mask = mask / (
        mask.max() + 1e-8
    )

    return mask

In [ ]:
def save_rollout_visualization(

    original_img,
    rollout_mask,
    save_path
):

    original_np = np.array(

        original_img.resize((224, 224))

    ).astype(np.uint8)

    # Heatmap
    heatmap = cv2.applyColorMap(

        np.uint8(255 * rollout_mask),

        cv2.COLORMAP_JET
    )

    heatmap = cv2.cvtColor(

        heatmap,

        cv2.COLOR_BGR2RGB
    )

    # Smooth overlay
    overlay = cv2.addWeighted(

        original_np,
        0.6,

        heatmap,
        0.4,

        0
    )

    cv2.imwrite(

        save_path,

        cv2.cvtColor(
            overlay,
            cv2.COLOR_RGB2BGR
        )
    )

In [ ]:
MAX_PER_CLASS = 5
MAX_MISCLASSIFIED = 10

correct_counts = {
    cls: 0
    for cls in class_names
}

misclassified_count = 0

In [ ]:
BASE_OUTPUT = "../outputs/vit/attention_rollout"

correct_dirs = {

    cls: os.path.join(
        BASE_OUTPUT,
        "correct",
        cls
    )

    for cls in class_names
}

misclassified_dir = os.path.join(

    BASE_OUTPUT,
    "misclassified"
)

MASK_DIR = os.path.join(
    BASE_OUTPUT,
    "masks"
)

# Create folders
for path in correct_dirs.values():

    os.makedirs(
        path,
        exist_ok=True
    )

os.makedirs(
    misclassified_dir,
    exist_ok=True
)

os.makedirs(
    MASK_DIR,
    exist_ok=True
)

print("Directories created.")

In [ ]:
TEST_DIR = r"../data/COVID_19_dataset/test"

test_transforms = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.5159, 0.5159, 0.5159),
        std=(0.2280, 0.2280, 0.2280)
    )
])

# Test dataset
test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transforms
)

# Test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Test loader created.")
print(f"Total test images: {len(test_dataset)}")

In [ ]:
model.eval()

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        preds = torch.argmax(
            outputs,
            dim=1
        )

        for i in range(len(images)):

            image_tensor = images[i].unsqueeze(0)

            true_idx = labels[i].item()
            pred_idx = preds[i].item()

            true_label = class_names[true_idx]
            pred_label = class_names[pred_idx]


            original_img = images[i].detach().cpu()

# Denormalize
            mean = torch.tensor(
                [0.5159, 0.5159, 0.5159]
            ).view(3,1,1)

            std = torch.tensor(
                [0.2280, 0.2280, 0.2280]
            ).view(3,1,1)

            original_img = (
                original_img * std
            ) + mean

            original_img = torch.clamp(
                original_img,
                0,
                1
           )

            original_img = original_img.permute(
                1,
                2,
                0
            ).numpy()

# Convert to uint8
            original_img = (
                original_img * 255
            ).astype(np.uint8)

        # Convert to PIL
            original_img = Image.fromarray(
            original_img
            )

            # ----------------------------------
            # Generate Rollout
            # ----------------------------------

            rollout_mask = generate_attention_rollout(

                model,
                image_tensor
            )
            mask_filename = (

            f"mask_true_{true_label}"
            f"_pred_{pred_label}"
            f"_{i}.npy"
            )

            mask_save_path = os.path.join(

            MASK_DIR,
            mask_filename
            )

            np.save(
            mask_save_path,
            rollout_mask
            )

            # ----------------------------------
            # Correct Predictions
            # ----------------------------------

            if true_idx == pred_idx:

                if (
                    correct_counts[true_label]
                    < MAX_PER_CLASS
                ):

                    save_path = os.path.join(

                        correct_dirs[true_label],

                        f"true_{true_label}"
                        f"_pred_{pred_label}"
                        f"_{correct_counts[true_label]}.png"
                    )

                    save_rollout_visualization(

                        original_img,
                        rollout_mask,
                        save_path,

                    )

                    correct_counts[
                        true_label
                    ] += 1

            # ----------------------------------
            # Misclassified
            # ----------------------------------

            else:

                if (
                    misclassified_count
                    < MAX_MISCLASSIFIED
                ):

                    save_path = os.path.join(

                        misclassified_dir,

                        f"true_{true_label}"
                        f"_pred_{pred_label}"
                        f"_{misclassified_count}.png"
                    )

                    save_rollout_visualization(

                        original_img,
                        rollout_mask,
                        save_path,
                    )

                    misclassified_count += 1

        # Stop condition
        if (

            all(

                count >= MAX_PER_CLASS

                for count in correct_counts.values()
            )

            and

            misclassified_count >= MAX_MISCLASSIFIED
        ):
            break

In [ ]:
print("\nSaved Correct Samples:\n")

for cls, count in correct_counts.items():

    print(f"{cls}: {count}")

print(
    f"\nSaved Misclassified: "
    f"{misclassified_count}"
)

print(
    "\nAttention rollout generation complete."
)

In [ ]:
mask_files = os.listdir(
    MASK_DIR
)

print(
    "Total masks:",
    len(mask_files)
)

sample_mask = np.load(

    os.path.join(
        MASK_DIR,
        mask_files[0]
    )
)

print(
    "Shape:",
    sample_mask.shape
)

print(
    "Min:",
    sample_mask.min()
)

print(
    "Max:",
    sample_mask.max()
)

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(
    sample_mask,
    cmap='jet'
)

plt.colorbar()

plt.title("Raw Rollout Mask")

plt.axis("off")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os
import cv2

SAVE_DIR = "../outputs/vit"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# Sample image
sample_tensor = images[0].detach().cpu()

# Denormalize
mean = torch.tensor(
    [0.5159, 0.5159, 0.5159]
).view(3,1,1)

std = torch.tensor(
    [0.2280, 0.2280, 0.2280]
).view(3,1,1)

sample_img = (
    sample_tensor * std
) + mean

sample_img = torch.clamp(
    sample_img,
    0,
    1
)

sample_img = sample_img.permute(
    1,
    2,
    0
).numpy()

# Generate overlay
heatmap = cv2.applyColorMap(

    np.uint8(255 * rollout_mask),

    cv2.COLORMAP_JET
)

heatmap = cv2.cvtColor(
    heatmap,
    cv2.COLOR_BGR2RGB
)

overlay = cv2.addWeighted(

    np.uint8(sample_img * 255),
    0.6,

    heatmap,
    0.4,

    0
)

# Plot
fig, ax = plt.subplots(
    1,
    3,
    figsize=(15,5)
)

ax[0].imshow(sample_img)
ax[0].set_title("Original")
ax[0].axis("off")

ax[1].imshow(
    rollout_mask,
    cmap="jet"
)

ax[1].set_title(
    "Attention Rollout"
)

ax[1].axis("off")

ax[2].imshow(overlay)

ax[2].set_title(
    "Overlay"
)

ax[2].axis("off")

plt.tight_layout()

save_path = os.path.join(

    SAVE_DIR,

    "attention_rollout.png"
)

plt.savefig(

    save_path,

    dpi=300,

    bbox_inches="tight"
)

plt.show()

print(f"Saved at: {save_path}")